In [1]:
# ================================================================
# CELL 1: KIỂM TRA & THỐNG KÊ MỨC ĐỘ ĐẦY ĐỦ CỦA DỮ LIỆU (DATA COMPLETENESS)
# Dữ liệu chuẩn hóa theo tác giả: 1 Bệnh nhân = 1 Dòng duy nhất (Không trùng lặp)
# ================================================================
import pandas as pd
import os

base_dir = r"../Data_Ready_For_Model"

df_a = pd.read_csv(os.path.join(base_dir, "Hospital_A", "private_A_data.csv"))
df_b = pd.read_csv(os.path.join(base_dir, "Hospital_B", "private_B_data.csv"))
df_s = pd.read_csv(os.path.join(base_dir, "Shared_Features", "shared_data.csv"))
df_label = pd.read_csv(os.path.join(base_dir, "labels.csv"))

feats_a = [c for c in df_a.columns if c != 'hadm_id']
feats_b = [c for c in df_b.columns if c != 'hadm_id']
feats_s = [c for c in df_s.columns if c != 'hadm_id']

# Đếm số dòng Full 100%
full_a = df_a[feats_a].notnull().all(axis=1).sum()
full_b = df_b[feats_b].notnull().all(axis=1).sum()
full_s = df_s[feats_s].notnull().all(axis=1).sum()

# Đếm số dòng có từ 50% chỉ số trở lên
ge_50_a = ((df_a[feats_a].notnull().sum(axis=1) / len(feats_a)) >= 0.5).sum()
ge_50_b = ((df_b[feats_b].notnull().sum(axis=1) / len(feats_b)) >= 0.5).sum()

# Đếm số dòng có ít nhất 1 chỉ số
any_a = df_a[feats_a].notnull().any(axis=1).sum()
any_b = df_b[feats_b].notnull().any(axis=1).sum()

df_completeness = pd.DataFrame({
    'Bộ dữ liệu': ['Bệnh viện A (9 sinh hiệu)', 'Bệnh viện B (14 xét nghiệm)', 'Shared Features (14 biến)'],
    'Tổng số bệnh nhân duy nhất': [f"{len(df_a):,}", f"{len(df_b):,}", f"{len(df_s):,}"],
    'Full 100% (Không thiếu)': [f"{full_a:,} ({full_a/len(df_a)*100:.2f}%)", f"{full_b:,} ({full_b/len(df_b)*100:.2f}%)", f"{full_s:,} ({full_s/len(df_s)*100:.2f}%)"],
    'Có >= 50% chỉ số': [f"{ge_50_a:,} ({ge_50_a/len(df_a)*100:.2f}%)", f"{ge_50_b:,} ({ge_50_b/len(df_b)*100:.2f}%)", "---"],
    'Có ít nhất 1 chỉ số': [f"{any_a:,} ({any_a/len(df_a)*100:.2f}%)", f"{any_b:,} ({any_b/len(df_b)*100:.2f}%)", f"{len(df_s):,} (100.0%)"]
})

print("=" * 100)
print("              BẢNG THỐNG KÊ MỨC ĐỘ ĐẦY ĐỦ CỦA DỮ LIỆU (CHUẨN TÁC GIẢ)")
print("=" * 100)
display(df_completeness)

print("\n" + "=" * 100)
vc = df_label['mortality'].value_counts()
print(f"• Tổng số bệnh nhân duy nhất: {len(df_label):,d} người")
print(f"• Số ca sống sót (Nhãn 0)   : {vc.get(0, 0):,d} ({vc.get(0, 0)/len(df_label)*100:.2f}%)")
print(f"• Số ca tử vong  (Nhãn 1)   : {vc.get(1, 0):,d} ({vc.get(1, 0)/len(df_label)*100:.2f}%)")
print(f"• Kiểm tra trùng lặp hadm_id: {'KHÔNG CÓ DÒNG NÀO BỊ TRÙNG (100% SẠCH)!' if df_label['hadm_id'].is_unique else 'Có dòng trùng!'}")
print("=" * 100)


              BẢNG THỐNG KÊ MỨC ĐỘ ĐẦY ĐỦ CỦA DỮ LIỆU (CHUẨN TÁC GIẢ)


,Bộ dữ liệu,Tổng số bệnh nhân duy nhất,Full 100% (Không thiếu),Có >= 50% chỉ số,Có ít nhất 1 chỉ số
0,Bệnh viện A (9 sinh hiệu),"223,452","20,360 (9.11%)","43,838 (19.62%)","43,858 (19.63%)"
1,Bệnh viện B (14 xét nghiệm),"223,452","11,025 (4.93%)","139,030 (62.22%)","155,494 (69.59%)"
2,Shared Features (14 biến),"223,452","1,292 (0.58%)",---,"223,452 (100.0%)"



• Tổng số bệnh nhân duy nhất: 223,452 người
• Số ca sống sót (Nhãn 0)   : 202,675 (90.70%)
• Số ca tử vong  (Nhãn 1)   : 20,777 (9.30%)
• Kiểm tra trùng lặp hadm_id: KHÔNG CÓ DÒNG NÀO BỊ TRÙNG (100% SẠCH)!


In [2]:
# ================================================================
# CELL 2: ĐẶC TRƯNG CHUNG (SHARED FEATURES) - 14 BIẾN CỐT LÕI CHUẨN TÁC GIẢ
# ================================================================
import pandas as pd
import os

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

base_dir = r"../Data_Ready_For_Model"
df_s = pd.read_csv(os.path.join(base_dir, "Shared_Features", "shared_data.csv"))
feats_s = [c for c in df_s.columns if c != 'hadm_id']

elix_dict = {
    'cardiovascular': 'Bệnh lý Tim mạch mãn tính (Suy tim sung huyết, loạn nhịp, tăng huyết áp)',
    'neurological': 'Bệnh lý Thần kinh mãn tính (Liệt nửa người, đột quỵ não cũ)',
    'pulmonary': 'Bệnh Phổi mãn tính (COPD, hen suyễn nặng, xơ phổi)',
    'diabetes': 'Bệnh Đái tháo đường (Có hoặc không kèm biến chứng)',
    'renal': 'Bệnh Suy thận mãn tính (Giảm mức lọc cầu thận mạn)',
    'liver': 'Bệnh Gan mãn tính (Xơ gan, suy tế bào gan)',
    'cancer': 'Khối u Ác tính / Ung thư di căn / U lympho',
    'mental_substance': 'Rối loạn Tâm thần hoặc Lạm dụng chất gây nghiện / Nghiện rượu',
    'hem_metabolic': 'Rối loạn Chuyển hóa & Huyết học (Thiếu máu mạn, béo phì)',
    'autoimmune': 'Bệnh lý Tự miễn dịch (Lupus ban đỏ, viêm khớp dạng thấp)'
}

shared_list = []
stt = 1
for f in feats_s:
    if f == 'gender':
        shared_list.append({'STT': stt, 'Phân Nhóm': '1. Nhân khẩu học', 'Tên Feature': f, 'Kiểu': '0: Nữ, 1: Nam', 'Ý nghĩa Lâm sàng & Y khoa': 'Giới tính sinh học bệnh nhân'})
    elif f == 'age':
        shared_list.append({'STT': stt, 'Phân Nhóm': '1. Nhân khẩu học', 'Tên Feature': f, 'Kiểu': 'Số nguyên (Tuổi)', 'Ý nghĩa Lâm sàng & Y khoa': 'Tuổi nhập viện (yếu tố nguy cơ hàng đầu trong ICU)'})
    elif f == 'gcs_min':
        shared_list.append({'STT': stt, 'Phân Nhóm': '2. Lâm sàng chung', 'Tên Feature': f, 'Kiểu': 'Điểm (3-15)', 'Ý nghĩa Lâm sàng & Y khoa': 'Thang điểm tri giác Glasgow thấp nhất trong 24h (3: Hôn mê sâu -> 15: Tỉnh táo)'})
    elif f == 'weight_kg':
        shared_list.append({'STT': stt, 'Phân Nhóm': '2. Lâm sàng chung', 'Tên Feature': f, 'Kiểu': 'Kilogram (kg)', 'Ý nghĩa Lâm sàng & Y khoa': 'Cân nặng dùng tính liều thuốc và dịch truyền hồi sức'})
    else:
        shared_list.append({'STT': stt, 'Phân Nhóm': '3. Bệnh nền Elixhauser (10 nhóm)', 'Tên Feature': f, 'Kiểu': 'Nhị phân (0/1)', 'Ý nghĩa Lâm sàng & Y khoa': elix_dict.get(f, f)})
    stt += 1

print("=" * 105)
print(f"1. BẢNG TỪ ĐIỂN GIẢI THÍCH CHI TIẾT TOÀN BỘ {len(shared_list)} ĐẶC TRƯNG CHUNG (SHARED FEATURES):")
print("=" * 105)
display(pd.DataFrame(shared_list))

print("\n" + "=" * 105)
print(f"2. HIỂN THỊ DỮ LIỆU MẪU 10 DÒNG ĐẦY ĐỦ TẤT CẢ {len(feats_s)} FEATURES CỦA NHÓM SHARED (KHÔNG BỊ CHE KHUẤT):")
print("=" * 105)
display(df_s[['hadm_id'] + feats_s].head(10))


1. BẢNG TỪ ĐIỂN GIẢI THÍCH CHI TIẾT TOÀN BỘ 14 ĐẶC TRƯNG CHUNG (SHARED FEATURES):


,STT,Phân Nhóm,Tên Feature,Kiểu,Ý nghĩa Lâm sàng & Y khoa
0,1,1. Nhân khẩu học,gender,"0: Nữ, 1: Nam",Giới tính sinh học bệnh nhân
1,2,1. Nhân khẩu học,age,Số nguyên (Tuổi),Tuổi nhập viện (yếu tố nguy cơ hàng đầu trong ICU)
2,3,3. Bệnh nền Elixhauser (10 nhóm),cardiovascular,Nhị phân (0/1),"Bệnh lý Tim mạch mãn tính (Suy tim sung huyết, loạn nhịp, tăng huyết áp)"
3,4,3. Bệnh nền Elixhauser (10 nhóm),neurological,Nhị phân (0/1),"Bệnh lý Thần kinh mãn tính (Liệt nửa người, đột quỵ não cũ)"
4,5,3. Bệnh nền Elixhauser (10 nhóm),pulmonary,Nhị phân (0/1),"Bệnh Phổi mãn tính (COPD, hen suyễn nặng, xơ phổi)"
5,6,3. Bệnh nền Elixhauser (10 nhóm),diabetes,Nhị phân (0/1),Bệnh Đái tháo đường (Có hoặc không kèm biến chứng)
6,7,3. Bệnh nền Elixhauser (10 nhóm),renal,Nhị phân (0/1),Bệnh Suy thận mãn tính (Giảm mức lọc cầu thận mạn)
7,8,3. Bệnh nền Elixhauser (10 nhóm),liver,Nhị phân (0/1),"Bệnh Gan mãn tính (Xơ gan, suy tế bào gan)"
8,9,3. Bệnh nền Elixhauser (10 nhóm),cancer,Nhị phân (0/1),Khối u Ác tính / Ung thư di căn / U lympho
9,10,3. Bệnh nền Elixhauser (10 nhóm),mental_substance,Nhị phân (0/1),Rối loạn Tâm thần hoặc Lạm dụng chất gây nghiện / Nghiện rượu



2. HIỂN THỊ DỮ LIỆU MẪU 10 DÒNG ĐẦY ĐỦ TẤT CẢ 14 FEATURES CỦA NHÓM SHARED (KHÔNG BỊ CHE KHUẤT):


,hadm_id,gender,age,cardiovascular,neurological,pulmonary,diabetes,renal,liver,cancer,mental_substance,hem_metabolic,autoimmune,gcs_min,weight_kg
0,22595853,0,52,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,NaN,NaN
1,25022803,0,19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,NaN,NaN
2,23052089,1,72,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,NaN,NaN
3,27250926,1,25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
4,22927623,0,55,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,NaN,NaN
5,22148160,1,60,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
6,20600184,1,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,NaN,NaN
7,25852320,1,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
8,28979390,0,53,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
9,26134563,0,74,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN


In [3]:
# ================================================================
# CELL 3: BỆNH VIỆN A (PRIVATE A - SINH HIỆU) - GIẢI THÍCH & XEM DỮ LIỆU 10 DÒNG
# ================================================================
import pandas as pd
import os

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

base_dir = r"../Data_Ready_For_Model"
df_a = pd.read_csv(os.path.join(base_dir, "Hospital_A", "private_A_data.csv"))
feats_a = [c for c in df_a.columns if c != 'hadm_id']

a_dict = [
    {'STT': 1, 'Tên Feature': 'heart_rate_mean', 'Đơn vị': 'bpm (nhịp/phút)', 'Ý nghĩa Y khoa': 'Nhịp tim trung bình 24h. Phản ứng sinh tồn khi sốc, mất máu hoặc sốt'},
    {'STT': 2, 'Tên Feature': 'sbp_mean', 'Đơn vị': 'mmHg', 'Ý nghĩa Y khoa': 'Huyết áp tâm thu trung bình (gộp cả NIBP và Động mạch), đo lực co bóp cơ tim'},
    {'STT': 3, 'Tên Feature': 'dbp_mean', 'Đơn vị': 'mmHg', 'Ý nghĩa Y khoa': 'Huyết áp tâm trương trung bình (Huyết áp tối thiểu), đo sức cản ngoại biên'},
    {'STT': 4, 'Tên Feature': 'mbp_mean', 'Đơn vị': 'mmHg', 'Ý nghĩa Y khoa': 'Huyết áp động mạch trung bình (MAP). MAP < 65 mmHg đe dọa ngừng tưới máu Não và Thận'},
    {'STT': 5, 'Tên Feature': 'resp_rate_mean', 'Đơn vị': 'lần/phút', 'Ý nghĩa Y khoa': 'Nhịp thở trung bình. Nhịp thở nhanh (> 30) cảnh báo suy hô hấp cấp (ARDS)'},
    {'STT': 6, 'Tên Feature': 'temp_c_mean', 'Đơn vị': 'độ C (°C)', 'Ý nghĩa Y khoa': 'Thân nhiệt trung bình (đã quy đổi độ F sang độ C chuẩn). Đánh giá sốt cao hoặc hạ nhiệt khi sốc'},
    {'STT': 7, 'Tên Feature': 'spo2_mean', 'Đơn vị': '%', 'Ý nghĩa Y khoa': 'Độ bão hòa oxy máu mao mạch qua đầu ngón tay. SpO2 < 90% tế bào thiếu oxy'},
    {'STT': 8, 'Tên Feature': 'fio2_max', 'Đơn vị': '% (21-100)', 'Ý nghĩa Y khoa': 'Tỷ lệ oxy thở máy cao nhất trong 24h. Càng cao tổn thương phổi càng nặng'},
    {'STT': 9, 'Tên Feature': 'urine_output_24h', 'Đơn vị': 'ml / 24h', 'Ý nghĩa Y khoa': 'Tổng lượng nước tiểu 24h đầu. Dưới 500ml là dấu hiệu suy thận cấp hoặc sốc tuần hoàn'}
]

print("=" * 105)
print(f"1. BẢNG TỪ ĐIỂN GIẢI THÍCH CHI TIẾT TOÀN BỘ {len(a_dict)} ĐẶC TRƯNG RIÊNG BỆNH VIỆN A (PRIVATE A):")
print("=" * 105)
display(pd.DataFrame(a_dict))

print("\n" + "=" * 105)
print(f"2. HIỂN THỊ DỮ LIỆU MẪU 10 DÒNG ĐẦY ĐỦ TẤT CẢ {len(feats_a)} FEATURES CỦA BỆNH VIỆN A (CÁC CA ĐẦY ĐỦ DỮ LIỆU):")
print("=" * 105)
sample_a = df_a.dropna().head(10) if df_a.dropna().shape[0] >= 10 else df_a.head(10)
display(sample_a[['hadm_id'] + feats_a])


1. BẢNG TỪ ĐIỂN GIẢI THÍCH CHI TIẾT TOÀN BỘ 9 ĐẶC TRƯNG RIÊNG BỆNH VIỆN A (PRIVATE A):


,STT,Tên Feature,Đơn vị,Ý nghĩa Y khoa
0,1,heart_rate_mean,bpm (nhịp/phút),"Nhịp tim trung bình 24h. Phản ứng sinh tồn khi sốc, mất máu hoặc sốt"
1,2,sbp_mean,mmHg,"Huyết áp tâm thu trung bình (gộp cả NIBP và Động mạch), đo lực co bóp cơ tim"
2,3,dbp_mean,mmHg,"Huyết áp tâm trương trung bình (Huyết áp tối thiểu), đo sức cản ngoại biên"
3,4,mbp_mean,mmHg,Huyết áp động mạch trung bình (MAP). MAP < 65 mmHg đe dọa ngừng tưới máu Não và Thận
4,5,resp_rate_mean,lần/phút,Nhịp thở trung bình. Nhịp thở nhanh (> 30) cảnh báo suy hô hấp cấp (ARDS)
5,6,temp_c_mean,độ C (°C),Thân nhiệt trung bình (đã quy đổi độ F sang độ C chuẩn). Đánh giá sốt cao hoặc hạ nhiệt khi sốc
6,7,spo2_mean,%,Độ bão hòa oxy máu mao mạch qua đầu ngón tay. SpO2 < 90% tế bào thiếu oxy
7,8,fio2_max,% (21-100),Tỷ lệ oxy thở máy cao nhất trong 24h. Càng cao tổn thương phổi càng nặng
8,9,urine_output_24h,ml / 24h,Tổng lượng nước tiểu 24h đầu. Dưới 500ml là dấu hiệu suy thận cấp hoặc sốc tuần hoàn



2. HIỂN THỊ DỮ LIỆU MẪU 10 DÒNG ĐẦY ĐỦ TẤT CẢ 9 FEATURES CỦA BỆNH VIỆN A (CÁC CA ĐẦY ĐỦ DỮ LIỆU):


,hadm_id,heart_rate_mean,sbp_mean,dbp_mean,mbp_mean,resp_rate_mean,temp_c_mean,spo2_mean,fio2_max,urine_output_24h
39,27793700,89.963636,101.528571,69.085714,79.642857,17.176471,36.975556,99.411765,30.0,2230.0
50,21329021,93.928571,114.481481,74.888889,86.884615,23.928571,36.808642,93.285714,100.0,4265.0
56,23197839,76.550000,112.545455,59.545455,86.636364,20.150000,36.972222,98.350000,100.0,2845.0
59,28094813,72.560000,108.076923,58.153846,73.923077,14.760000,36.388889,98.000000,100.0,3875.0
69,26048429,107.700000,105.890909,72.363636,80.727273,11.600000,36.240741,94.500000,100.0,2230.0
95,21255400,85.327273,98.000000,51.950820,67.145161,12.709091,36.616162,99.933333,100.0,3810.0
97,29242151,94.652174,153.961538,61.653846,87.461538,18.739130,37.698413,98.304348,40.0,2215.0
102,22081550,73.955556,103.415094,57.377358,68.660377,15.000000,35.200000,98.094340,60.0,755.0
103,27411876,95.352941,147.878788,78.272727,93.636364,21.705882,37.817460,95.735294,50.0,3665.0
105,24817563,89.533333,107.837838,48.486486,67.594595,18.633333,36.420635,95.064516,100.0,2825.0


In [4]:
# ================================================================
# CELL 4: BỆNH VIỆN B (PRIVATE B - XÉT NGHIỆM LAB) - GIẢI THÍCH & XEM DỮ LIỆU 10 DÒNG
# ================================================================
import pandas as pd
import os

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

base_dir = r"../Data_Ready_For_Model"
df_b = pd.read_csv(os.path.join(base_dir, "Hospital_B", "private_B_data.csv"))
feats_b = [c for c in df_b.columns if c != 'hadm_id']

b_dict = [
    {'STT': 1, 'Nhóm Xét Nghiệm': 'Thận & Chuyển hóa', 'Tên Feature': 'creatinine_max', 'Đơn vị': 'mg/dL', 'Ý nghĩa Y khoa': 'Creatinine máu cao nhất trong 24h. Tiêu chuẩn vàng đánh giá Suy Thận Cấp (AKI)'},
    {'STT': 2, 'Nhóm Xét Nghiệm': 'Thận & Chuyển hóa', 'Tên Feature': 'bun_max', 'Đơn vị': 'mg/dL', 'Ý nghĩa Y khoa': 'Ure máu cao nhất (BUN), đánh giá sự tích tụ độc chất nitơ do suy thận'},
    {'STT': 3, 'Nhóm Xét Nghiệm': 'Khí Máu & Toan Kiềm', 'Tên Feature': 'anion_gap', 'Đơn vị': 'mEq/L', 'Ý nghĩa Y khoa': 'Khoảng trống Anion máu, phân loại toan chuyển hóa (toan lactic, toan ceton)'},
    {'STT': 4, 'Nhóm Xét Nghiệm': 'Khí Máu & Toan Kiềm', 'Tên Feature': 'lactate_max', 'Đơn vị': 'mmol/L', 'Ý nghĩa Y khoa': 'Nồng độ Axit Lactic cao nhất. CHỈ BÁO TỬ VONG SỐ 1 TRONG ICU khi tế bào thiếu máu oxy'},
    {'STT': 5, 'Nhóm Xét Nghiệm': 'Khí Máu & Toan Kiềm', 'Tên Feature': 'ph_min', 'Đơn vị': 'Thang pH', 'Ý nghĩa Y khoa': 'Độ pH máu thấp nhất. pH < 7.20 là toan máu nặng đe dọa ngừng tim'},
    {'STT': 6, 'Nhóm Xét Nghiệm': 'Điện Giải Đồ', 'Tên Feature': 'potassium_mean', 'Đơn vị': 'mEq/L (K+)', 'Ý nghĩa Y khoa': 'Kali máu trung bình. Tăng hoặc hạ Kali đều gây loạn nhịp thất, ngừng tim đột ngột'},
    {'STT': 7, 'Nhóm Xét Nghiệm': 'Điện Giải Đồ', 'Tên Feature': 'sodium_mean', 'Đơn vị': 'mEq/L (Na+)', 'Ý nghĩa Y khoa': 'Natri máu trung bình, quyết định áp suất thẩm thấu và thăng bằng thể tích dịch'},
    {'STT': 8, 'Nhóm Xét Nghiệm': 'Điện Giải Đồ', 'Tên Feature': 'chloride_mean', 'Đơn vị': 'mEq/L (Cl-)', 'Ý nghĩa Y khoa': 'Clo máu trung bình, tham gia cân bằng toan kiềm và điện tích với Na+'},
    {'STT': 9, 'Nhóm Xét Nghiệm': 'Huyết Học & Viêm', 'Tên Feature': 'wbc_max', 'Đơn vị': 'K/uL (x1000)', 'Ý nghĩa Y khoa': 'Bạch cầu cao nhất. WBC > 12.000 hoặc < 4.000 báo hiệu Nhiễm Trùng Huyết nặng'},
    {'STT': 10, 'Nhóm Xét Nghiệm': 'Huyết Học & Mất Máu', 'Tên Feature': 'hemoglobin_min', 'Đơn vị': 'g/dL', 'Ý nghĩa Y khoa': 'Huyết sắc tố thấp nhất trong 24h, đánh giá mức độ mất máu cấp / thiếu máu nặng'},
    {'STT': 11, 'Nhóm Xét Nghiệm': 'Huyết Học & Đông Máu', 'Tên Feature': 'platelets_min', 'Đơn vị': 'K/uL (x1000)', 'Ý nghĩa Y khoa': 'Tiểu cầu thấp nhất. Tiểu cầu giảm sâu (< 50.000) cảnh báo xuất huyết ồ ạt (DIC)'},
    {'STT': 12, 'Nhóm Xét Nghiệm': 'Huyết Học & Đông Máu', 'Tên Feature': 'inr_max', 'Đơn vị': 'Chỉ số INR', 'Ý nghĩa Y khoa': 'Tỷ lệ đông máu ngoại sinh INR. INR cao (> 1.5) cảnh báo suy gan cấp / rối loạn đông máu'},
    {'STT': 13, 'Nhóm Xét Nghiệm': 'Gan Mật & Chuyển Hóa', 'Tên Feature': 'glucose_mean', 'Đơn vị': 'mg/dL', 'Ý nghĩa Y khoa': 'Đường huyết trung bình. Căng thẳng chuyển hóa cấp hoặc hạ đường huyết nguy kịch'},
    {'STT': 14, 'Nhóm Xét Nghiệm': 'Gan Mật & Chuyển Hóa', 'Tên Feature': 'bilirubin_max', 'Đơn vị': 'mg/dL', 'Ý nghĩa Y khoa': 'Men sắc tố mật Bilirubin cao nhất. Đánh giá suy chức năng tế bào gan, tắc mật, vàng da'}
]

print("=" * 105)
print(f"1. BẢNG TỪ ĐIỂN GIẢI THÍCH CHI TIẾT TOÀN BỘ {len(b_dict)} ĐẶC TRƯNG RIÊNG BỆNH VIỆN B (PRIVATE B):")
print("=" * 105)
display(pd.DataFrame(b_dict))

print("\n" + "=" * 105)
print(f"2. HIỂN THỊ DỮ LIỆU MẪU 10 DÒNG ĐẦY ĐỦ TẤT CẢ {len(feats_b)} FEATURES CỦA BỆNH VIỆN B (CÁC CA ĐẦY ĐỦ DỮ LIỆU):")
print("=" * 105)
sample_b = df_b.dropna().head(10) if df_b.dropna().shape[0] >= 10 else df_b.head(10)
display(sample_b[['hadm_id'] + feats_b])


1. BẢNG TỪ ĐIỂN GIẢI THÍCH CHI TIẾT TOÀN BỘ 14 ĐẶC TRƯNG RIÊNG BỆNH VIỆN B (PRIVATE B):


,STT,Nhóm Xét Nghiệm,Tên Feature,Đơn vị,Ý nghĩa Y khoa
0,1,Thận & Chuyển hóa,creatinine_max,mg/dL,Creatinine máu cao nhất trong 24h. Tiêu chuẩn vàng đánh giá Suy Thận Cấp (AKI)
1,2,Thận & Chuyển hóa,bun_max,mg/dL,"Ure máu cao nhất (BUN), đánh giá sự tích tụ độc chất nitơ do suy thận"
2,3,Khí Máu & Toan Kiềm,anion_gap,mEq/L,"Khoảng trống Anion máu, phân loại toan chuyển hóa (toan lactic, toan ceton)"
3,4,Khí Máu & Toan Kiềm,lactate_max,mmol/L,Nồng độ Axit Lactic cao nhất. CHỈ BÁO TỬ VONG SỐ 1 TRONG ICU khi tế bào thiếu máu oxy
4,5,Khí Máu & Toan Kiềm,ph_min,Thang pH,Độ pH máu thấp nhất. pH < 7.20 là toan máu nặng đe dọa ngừng tim
5,6,Điện Giải Đồ,potassium_mean,mEq/L (K+),"Kali máu trung bình. Tăng hoặc hạ Kali đều gây loạn nhịp thất, ngừng tim đột ngột"
6,7,Điện Giải Đồ,sodium_mean,mEq/L (Na+),"Natri máu trung bình, quyết định áp suất thẩm thấu và thăng bằng thể tích dịch"
7,8,Điện Giải Đồ,chloride_mean,mEq/L (Cl-),"Clo máu trung bình, tham gia cân bằng toan kiềm và điện tích với Na+"
8,9,Huyết Học & Viêm,wbc_max,K/uL (x1000),Bạch cầu cao nhất. WBC > 12.000 hoặc < 4.000 báo hiệu Nhiễm Trùng Huyết nặng
9,10,Huyết Học & Mất Máu,hemoglobin_min,g/dL,"Huyết sắc tố thấp nhất trong 24h, đánh giá mức độ mất máu cấp / thiếu máu nặng"



2. HIỂN THỊ DỮ LIỆU MẪU 10 DÒNG ĐẦY ĐỦ TẤT CẢ 14 FEATURES CỦA BỆNH VIỆN B (CÁC CA ĐẦY ĐỦ DỮ LIỆU):


,hadm_id,creatinine_max,bun_max,anion_gap,lactate_max,ph_min,potassium_mean,sodium_mean,chloride_mean,wbc_max,hemoglobin_min,platelets_min,inr_max,glucose_mean,bilirubin_max
39,27793700,2.7,44.0,17.0,5.2,7.54,3.350000,128.250000,76.000000,8.4,10.1,98.0,2.0,118.250000,1.1
50,21329021,0.9,24.0,12.0,1.9,7.27,4.900000,133.333333,100.333333,20.7,14.4,307.0,1.3,230.333333,0.5
56,23197839,0.7,10.0,13.0,3.5,7.10,4.000000,140.000000,106.000000,16.7,9.1,198.0,1.1,93.000000,0.5
90,24181354,7.9,92.0,28.0,4.5,7.10,4.133333,132.000000,96.666667,21.6,11.9,12.0,1.3,289.666667,6.5
97,29242151,0.9,11.0,20.0,1.0,7.44,3.700000,139.500000,99.500000,14.1,12.5,166.0,1.1,131.000000,1.7
102,22081550,0.3,15.0,14.0,2.2,7.38,4.000000,135.000000,99.333333,16.0,10.6,481.0,1.5,136.333333,0.4
105,24817563,1.2,20.0,14.0,3.7,7.32,4.100000,139.000000,105.500000,17.5,9.8,104.0,1.5,174.000000,0.6
125,29646384,0.8,6.0,19.0,3.9,7.34,4.450000,142.500000,101.500000,24.9,8.7,145.0,1.3,152.000000,1.3
138,22942076,2.0,18.0,32.0,12.0,7.04,5.400000,148.333333,110.333333,12.0,8.1,42.0,2.6,179.666667,3.2
171,22943998,1.6,46.0,15.0,1.3,7.39,4.200000,133.000000,97.000000,9.7,16.3,178.0,1.1,166.000000,0.7
